# 🏆 Kaggle Success Factors: Exploratory Data Analysis

## What Makes Kaggle Content Go Viral? 100K Notebooks & Datasets Decoded

This notebook explores the success factors behind **100,000 Kaggle notebooks and datasets**, uncovering patterns in engagement, medals, and virality.

### Table of Contents
1. [Data Loading & Overview](#1)
2. [Content Distribution Analysis](#2)
3. [Engagement Metrics Analysis](#3)
4. [The Author Tier Effect](#4)
5. [Medal Success Factors](#5)
6. [Temporal Analysis](#6)
7. [Topic Analysis](#7)
8. [Correlation Analysis](#8)
9. [Notebook-Specific Insights](#9)
10. [Dataset-Specific Insights](#10)
11. [Key Findings Summary](#11)

---
## 📦 Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings

warnings.filterwarnings('ignore')

# Visualization settings
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

# Color palettes
MEDAL_COLORS = {'Gold': '#FFD700', 'Silver': '#C0C0C0', 'Bronze': '#CD7F32', 'None': '#E8E8E8'}
TIER_COLORS = {'Grandmaster': '#FF4444', 'Master': '#FF8800', 'Expert': '#9933FF', 
               'Contributor': '#00AAFF', 'Novice': '#00CC66'}
CONTENT_COLORS = {'notebook': '#4ECDC4', 'dataset': '#FF6B6B'}

print('Libraries loaded successfully!')

<a id='1'></a>
---
## 📂 1. Data Loading & Overview

In [ ]:
# Load the dataset
df = pd.read_csv('/kaggle/input/kaggle-success-factors/kaggle_success_factors.csv')

print(f'Dataset Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Memory Usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB')

In [ ]:
# Preview the data
df.head()

In [ ]:
# Data types overview
df.info()

In [ ]:
# Missing values analysis
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Count': missing, 'Percentage': missing_pct})
print('Missing Values (columns with nulls):')
missing_df[missing_df['Count'] > 0].sort_values('Count', ascending=False)

In [ ]:
# Statistical summary
df.describe()

<a id='2'></a>
---
## 📊 2. Content Distribution Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 2.1 Content Type Distribution
ax1 = axes[0, 0]
content_counts = df['content_type'].value_counts()
colors = [CONTENT_COLORS[x] for x in content_counts.index]
ax1.pie(content_counts, labels=content_counts.index, autopct='%1.1f%%', 
        colors=colors, explode=[0.02, 0.02], startangle=90)
ax1.set_title('Content Type Distribution', fontweight='bold')

# 2.2 Programming Language Distribution
ax2 = axes[0, 1]
lang_counts = df['programming_language'].value_counts()
bars = ax2.bar(lang_counts.index, lang_counts.values, color=['#3776AB', '#276DC3', '#9558B2', '#336791'])
ax2.set_title('Programming Language Distribution', fontweight='bold')
ax2.set_ylabel('Count')
for bar, val in zip(bars, lang_counts.values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500, 
             f'{val:,}', ha='center', va='bottom', fontsize=10)

# 2.3 Author Tier Distribution
ax3 = axes[1, 0]
tier_order = ['Grandmaster', 'Master', 'Expert', 'Contributor', 'Novice']
tier_counts = df['author_tier'].value_counts().reindex(tier_order)
colors = [TIER_COLORS[t] for t in tier_order]
bars = ax3.barh(tier_order[::-1], tier_counts.values[::-1], color=colors[::-1])
ax3.set_title('Author Tier Distribution', fontweight='bold')
ax3.set_xlabel('Count')

# 2.4 Medal Distribution
ax4 = axes[1, 1]
medal_order = ['Gold', 'Silver', 'Bronze', 'None']
medal_counts = df['medal'].value_counts().reindex(medal_order)
colors = [MEDAL_COLORS[m] for m in medal_order]
ax4.pie(medal_counts, labels=medal_order, autopct='%1.1f%%',
        colors=colors, explode=[0.1, 0.05, 0.02, 0])
ax4.set_title('Medal Distribution', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
print('Key Insights:')
print(f'  - {content_counts["notebook"]:,} notebooks ({content_counts["notebook"]/len(df)*100:.1f}%)')
print(f'  - {content_counts["dataset"]:,} datasets ({content_counts["dataset"]/len(df)*100:.1f}%)')
print(f'  - Python dominates with {lang_counts["Python"]:,} entries ({lang_counts["Python"]/len(df)*100:.1f}%)')
print(f'  - Only {medal_counts["Gold"]:,} Gold medals ({medal_counts["Gold"]/len(df)*100:.2f}%)')

<a id='3'></a>
---
## 📈 3. Engagement Metrics Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 3.1 Views Distribution (Log Scale)
ax1 = axes[0, 0]
ax1.hist(np.log10(df['views'] + 1), bins=50, color='#4ECDC4', edgecolor='white', alpha=0.8)
ax1.set_title('Views Distribution (Log Scale)', fontweight='bold')
ax1.set_xlabel('log10(Views)')
ax1.set_ylabel('Frequency')
ax1.axvline(np.log10(df['views'].median()), color='red', linestyle='--', 
            label=f"Median: {df['views'].median():,.0f}")
ax1.legend()

# 3.2 Upvotes Distribution (Log Scale)
ax2 = axes[0, 1]
ax2.hist(np.log10(df['upvotes'] + 1), bins=50, color='#FF6B6B', edgecolor='white', alpha=0.8)
ax2.set_title('Upvotes Distribution (Log Scale)', fontweight='bold')
ax2.set_xlabel('log10(Upvotes)')
ax2.set_ylabel('Frequency')
ax2.axvline(np.log10(df['upvotes'].median() + 1), color='red', linestyle='--', 
            label=f"Median: {df['upvotes'].median():,.0f}")
ax2.legend()

# 3.3 Views vs Upvotes Scatter
ax3 = axes[1, 0]
sample = df.sample(min(5000, len(df)), random_state=42)
scatter = ax3.scatter(np.log10(sample['views'] + 1), np.log10(sample['upvotes'] + 1), 
                      c=sample['quality_score'], cmap='viridis', alpha=0.5, s=10)
ax3.set_title('Views vs Upvotes (colored by Quality Score)', fontweight='bold')
ax3.set_xlabel('log10(Views)')
ax3.set_ylabel('log10(Upvotes)')
plt.colorbar(scatter, ax=ax3, label='Quality Score')

# 3.4 Engagement Rate by Content Type
ax4 = axes[1, 1]
df.boxplot(column='engagement_rate', by='content_type', ax=ax4)
ax4.set_title('Engagement Rate by Content Type', fontweight='bold')
ax4.set_xlabel('Content Type')
ax4.set_ylabel('Engagement Rate')
plt.suptitle('')

plt.tight_layout()
plt.show()

In [ ]:
# Engagement Statistics
print('Engagement Statistics:')
df[['views', 'upvotes', 'comments_count', 'engagement_rate']].describe().T[['mean', '50%', 'max']].round(2)

<a id='4'></a>
---
## 👤 4. The Author Tier Effect

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

tier_order = ['Novice', 'Contributor', 'Expert', 'Master', 'Grandmaster']

# 4.1 Average Views by Author Tier
ax1 = axes[0, 0]
tier_views = df.groupby('author_tier')['views'].mean().reindex(tier_order)
colors = [TIER_COLORS[t] for t in tier_order]
bars = ax1.bar(tier_order, tier_views.values, color=colors)
ax1.set_title('Average Views by Author Tier', fontweight='bold')
ax1.set_ylabel('Average Views')
ax1.set_xticklabels(tier_order, rotation=45, ha='right')
for bar, val in zip(bars, tier_views.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50, 
             f'{val:,.0f}', ha='center', va='bottom', fontsize=9)

# 4.2 Average Upvotes by Author Tier
ax2 = axes[0, 1]
tier_upvotes = df.groupby('author_tier')['upvotes'].mean().reindex(tier_order)
bars = ax2.bar(tier_order, tier_upvotes.values, color=colors)
ax2.set_title('Average Upvotes by Author Tier', fontweight='bold')
ax2.set_ylabel('Average Upvotes')
ax2.set_xticklabels(tier_order, rotation=45, ha='right')

# 4.3 Medal Rate by Author Tier
ax3 = axes[1, 0]
tier_medal = df[df['medal'] != 'None'].groupby('author_tier').size() / df.groupby('author_tier').size() * 100
tier_medal = tier_medal.reindex(tier_order).fillna(0)
bars = ax3.bar(tier_order, tier_medal.values, color=colors)
ax3.set_title('Medal Rate by Author Tier (%)', fontweight='bold')
ax3.set_ylabel('Medal Rate (%)')
ax3.set_xticklabels(tier_order, rotation=45, ha='right')
for bar, val in zip(bars, tier_medal.values):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
             f'{val:.1f}%', ha='center', va='bottom', fontsize=9)

# 4.4 Author Followers Distribution by Tier
ax4 = axes[1, 1]
for tier in tier_order:
    tier_data = df[df['author_tier'] == tier]['author_followers']
    ax4.hist(np.log10(tier_data + 1), bins=30, alpha=0.5, label=tier, color=TIER_COLORS[tier])
ax4.set_title('Follower Distribution by Tier (Log Scale)', fontweight='bold')
ax4.set_xlabel('log10(Followers)')
ax4.set_ylabel('Frequency')
ax4.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Tier Impact Analysis
gm_avg = tier_views['Grandmaster']
novice_avg = tier_views['Novice']
print('Tier Impact Analysis:')
print(f'  - Grandmasters get {gm_avg/novice_avg:.1f}x more views than Novices')
print(f'  - Grandmaster medal rate: {tier_medal["Grandmaster"]:.1f}%')
print(f'  - Novice medal rate: {tier_medal["Novice"]:.1f}%')

<a id='5'></a>
---
## 🏅 5. Medal Success Factors

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

medal_order = ['None', 'Bronze', 'Silver', 'Gold']

# 5.1 Quality Score by Medal
ax1 = axes[0, 0]
df.boxplot(column='quality_score', by='medal', ax=ax1)
ax1.set_title('Quality Score by Medal', fontweight='bold')
ax1.set_xlabel('Medal')
ax1.set_ylabel('Quality Score')
plt.suptitle('')

# 5.2 Views by Medal (Log Scale)
ax2 = axes[0, 1]
medal_views = df.groupby('medal')['views'].median().reindex(medal_order)
colors = [MEDAL_COLORS[m] for m in medal_order]
bars = ax2.bar(medal_order, medal_views.values, color=colors, edgecolor='black')
ax2.set_title('Median Views by Medal', fontweight='bold')
ax2.set_ylabel('Median Views')
ax2.set_yscale('log')
for bar, val in zip(bars, medal_views.values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.1, 
             f'{val:,.0f}', ha='center', va='bottom', fontsize=9)

# 5.3 Competition Effect on Medals
ax3 = axes[1, 0]
comp_medal = pd.crosstab(df['is_competition_related'], df['medal'], normalize='index') * 100
comp_medal = comp_medal[['Gold', 'Silver', 'Bronze', 'None']]
comp_medal.plot(kind='bar', stacked=True, ax=ax3, 
                color=[MEDAL_COLORS[m] for m in ['Gold', 'Silver', 'Bronze', 'None']])
ax3.set_title('Medal Distribution: Competition vs Non-Competition', fontweight='bold')
ax3.set_xlabel('Competition Related')
ax3.set_ylabel('Percentage (%)')
ax3.set_xticklabels(['No', 'Yes'], rotation=0)
ax3.legend(title='Medal', bbox_to_anchor=(1.02, 1))

# 5.4 Visualization Count Impact (Notebooks Only)
ax4 = axes[1, 1]
notebooks = df[df['content_type'] == 'notebook'].copy()
notebooks['viz_bucket'] = pd.cut(notebooks['visualization_count'], 
                                  bins=[0, 5, 10, 20, 50, 100], 
                                  labels=['0-5', '6-10', '11-20', '21-50', '51+'])
viz_medal = notebooks[notebooks['medal'] != 'None'].groupby('viz_bucket').size() / notebooks.groupby('viz_bucket').size() * 100
bars = ax4.bar(viz_medal.index.astype(str), viz_medal.values, color='#9B59B6', edgecolor='white')
ax4.set_title('Medal Rate by Visualization Count (Notebooks)', fontweight='bold')
ax4.set_xlabel('Visualization Count')
ax4.set_ylabel('Medal Rate (%)')

plt.tight_layout()
plt.show()

In [ ]:
# Medal Statistics
comp_related = df[df['is_competition_related'] == True]
non_comp = df[df['is_competition_related'] == False]
comp_medal_rate = (comp_related['medal'] != 'None').mean() * 100
non_comp_medal_rate = (non_comp['medal'] != 'None').mean() * 100

print('Medal Success Insights:')
print(f'  - Competition-related medal rate: {comp_medal_rate:.2f}%')
print(f'  - Non-competition medal rate: {non_comp_medal_rate:.2f}%')
print(f'  - Competition content is {comp_medal_rate/non_comp_medal_rate:.1f}x more likely to earn medals')

<a id='6'></a>
---
## ⏰ 6. Temporal Analysis

In [ ]:
# Convert dates
df['created_date'] = pd.to_datetime(df['created_date'])
df['year_month'] = df['created_date'].dt.to_period('M')
df['year'] = df['created_date'].dt.year

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 6.1 Content Creation Over Time
ax1 = axes[0, 0]
monthly_counts = df.groupby('year_month').size()
ax1.plot(monthly_counts.index.astype(str), monthly_counts.values, color='#3498DB', linewidth=2)
ax1.fill_between(range(len(monthly_counts)), monthly_counts.values, alpha=0.3, color='#3498DB')
ax1.set_title('Content Creation Over Time', fontweight='bold')
ax1.set_xlabel('Month')
ax1.set_ylabel('Count')
ax1.set_xticks(range(0, len(monthly_counts), 12))
ax1.set_xticklabels([str(monthly_counts.index[i]) for i in range(0, len(monthly_counts), 12)], rotation=45, ha='right')

# 6.2 Average Engagement by Year
ax2 = axes[0, 1]
yearly_eng = df.groupby('year')[['views', 'upvotes']].mean()
ax2.bar(yearly_eng.index - 0.2, yearly_eng['views'], width=0.4, label='Avg Views', color='#4ECDC4')
ax2.bar(yearly_eng.index + 0.2, yearly_eng['upvotes'] * 100, width=0.4, label='Avg Upvotes (x100)', color='#FF6B6B')
ax2.set_title('Average Engagement by Year', fontweight='bold')
ax2.set_xlabel('Year')
ax2.set_ylabel('Count')
ax2.legend()

# 6.3 Age vs Engagement
ax3 = axes[1, 0]
age_bins = pd.cut(df['days_since_creation'], bins=[0, 90, 180, 365, 730, 2500], 
                  labels=['<3mo', '3-6mo', '6-12mo', '1-2yr', '2+yr'])
age_views = df.groupby(age_bins, observed=True)['views'].median()
bars = ax3.bar(age_views.index.astype(str), age_views.values, color='#9B59B6', edgecolor='white')
ax3.set_title('Median Views by Content Age', fontweight='bold')
ax3.set_xlabel('Content Age')
ax3.set_ylabel('Median Views')

# 6.4 Update Frequency Impact
ax4 = axes[1, 1]
df['is_updated'] = df['update_count'] > 0
update_medal = df.groupby('is_updated')['medal'].apply(lambda x: (x != 'None').mean() * 100)
bars = ax4.bar(['Never Updated', 'Updated'], update_medal.values, color=['#E74C3C', '#27AE60'])
ax4.set_title('Medal Rate: Updated vs Never Updated', fontweight='bold')
ax4.set_ylabel('Medal Rate (%)')
for bar, val in zip(bars, update_medal.values):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2, 
             f'{val:.1f}%', ha='center', va='bottom', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
print('Temporal Insights:')
print(f'  - Older content accumulates more views (2+ years median: {age_views["2+yr"]:,.0f})')
print(f'  - Updated content medal rate: {update_medal[True]:.1f}%')
print(f'  - Never updated medal rate: {update_medal[False]:.1f}%')

<a id='7'></a>
---
## 🏷️ 7. Topic Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 7.1 Top 15 Topics by Count
ax1 = axes[0]
topic_counts = df['primary_topic'].value_counts().head(15)
bars = ax1.barh(topic_counts.index[::-1], topic_counts.values[::-1], color='#3498DB', edgecolor='white')
ax1.set_title('Top 15 Topics by Volume', fontweight='bold')
ax1.set_xlabel('Count')

# 7.2 Top Topics by Medal Rate
ax2 = axes[1]
topic_medal = df.groupby('primary_topic').apply(lambda x: (x['medal'] != 'None').mean() * 100)
topic_medal = topic_medal[df['primary_topic'].value_counts() > 500].sort_values(ascending=False).head(15)
bars = ax2.barh(topic_medal.index[::-1], topic_medal.values[::-1], color='#E74C3C', edgecolor='white')
ax2.set_title('Top 15 Topics by Medal Rate', fontweight='bold')
ax2.set_xlabel('Medal Rate (%)')

plt.tight_layout()
plt.show()

<a id='8'></a>
---
## 🔗 8. Correlation Analysis

In [ ]:
# Select numeric columns for correlation
numeric_cols = ['views', 'upvotes', 'comments_count', 'fork_count', 'author_followers',
                'author_notebooks_count', 'days_since_creation', 'engagement_rate', 
                'virality_score', 'quality_score']

corr_matrix = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, cmap='RdBu_r', center=0,
            fmt='.2f', square=True, linewidths=0.5, ax=ax,
            cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Matrix of Key Metrics', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Top Correlations with Upvotes
print('Top Correlations with Upvotes:')
upvote_corr = corr_matrix['upvotes'].drop('upvotes').sort_values(ascending=False)
for col, val in upvote_corr.head(5).items():
    print(f'  - {col}: {val:.3f}')

<a id='9'></a>
---
## 📓 9. Notebook-Specific Insights

In [ ]:
notebooks = df[df['content_type'] == 'notebook'].copy()
print(f'Total Notebooks: {len(notebooks):,}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 9.1 Code Length Distribution
ax1 = axes[0, 0]
ax1.hist(np.log10(notebooks['code_lines'] + 1), bins=50, color='#3498DB', edgecolor='white', alpha=0.8)
ax1.set_title('Code Length Distribution (Log Scale)', fontweight='bold')
ax1.set_xlabel('log10(Code Lines)')
ax1.set_ylabel('Frequency')
ax1.axvline(np.log10(notebooks['code_lines'].median()), color='red', linestyle='--', 
            label=f"Median: {notebooks['code_lines'].median():.0f}")
ax1.legend()

# 9.2 Markdown Ratio vs Medal
ax2 = axes[0, 1]
notebooks.boxplot(column='markdown_ratio', by='medal', ax=ax2)
ax2.set_title('Markdown Ratio by Medal', fontweight='bold')
ax2.set_xlabel('Medal')
ax2.set_ylabel('Markdown Ratio')
plt.suptitle('')

# 9.3 GPU Usage Impact
ax3 = axes[1, 0]
gpu_stats = notebooks.groupby('uses_gpu').agg({'upvotes': 'mean', 'views': 'mean'}).reset_index()
x = np.arange(2)
width = 0.35
ax3.bar(x - width/2, gpu_stats['views'], width, label='Avg Views', color='#4ECDC4')
ax3.bar(x + width/2, gpu_stats['upvotes'] * 100, width, label='Avg Upvotes (x100)', color='#FF6B6B')
ax3.set_xticks(x)
ax3.set_xticklabels(['No GPU', 'Uses GPU'])
ax3.set_title('GPU Usage Impact on Engagement', fontweight='bold')
ax3.legend()

# 9.4 Top Libraries
ax4 = axes[1, 1]
all_libs = notebooks['libraries_used'].dropna().str.split('|').explode()
lib_counts = all_libs.value_counts().head(12)
bars = ax4.barh(lib_counts.index[::-1], lib_counts.values[::-1], color='#9B59B6', edgecolor='white')
ax4.set_title('Top 12 Libraries Used', fontweight='bold')
ax4.set_xlabel('Count')

plt.tight_layout()
plt.show()

<a id='10'></a>
---
## 📦 10. Dataset-Specific Insights

In [ ]:
datasets = df[df['content_type'] == 'dataset'].copy()
print(f'Total Datasets: {len(datasets):,}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 10.1 Usability Score Distribution
ax1 = axes[0, 0]
ax1.hist(datasets['usability_score'].dropna(), bins=20, color='#E74C3C', edgecolor='white', alpha=0.8)
ax1.set_title('Usability Score Distribution', fontweight='bold')
ax1.set_xlabel('Usability Score')
ax1.set_ylabel('Frequency')

# 10.2 File Format Distribution
ax2 = axes[0, 1]
format_counts = datasets['file_format'].value_counts()
ax2.pie(format_counts, labels=format_counts.index, autopct='%1.1f%%', startangle=90)
ax2.set_title('File Format Distribution', fontweight='bold')

# 10.3 Usability Score vs Downloads
ax3 = axes[1, 0]
scatter = ax3.scatter(datasets['usability_score'], np.log10(datasets['downloads'] + 1),
                      c=datasets['notebook_usage'], cmap='viridis', alpha=0.5, s=20)
ax3.set_title('Usability Score vs Downloads', fontweight='bold')
ax3.set_xlabel('Usability Score')
ax3.set_ylabel('log10(Downloads)')
plt.colorbar(scatter, ax=ax3, label='Notebook Usage')

# 10.4 License Type Distribution
ax4 = axes[1, 1]
license_counts = datasets['license_type'].value_counts()
bars = ax4.barh(license_counts.index[::-1], license_counts.values[::-1], color='#27AE60', edgecolor='white')
ax4.set_title('License Type Distribution', fontweight='bold')
ax4.set_xlabel('Count')

plt.tight_layout()
plt.show()

<a id='11'></a>
---
## 🏆 11. Key Findings Summary

In [ ]:
print('='*60)
print('KEY FINDINGS SUMMARY')
print('='*60)

print('''
CONTENT DISTRIBUTION:
  - 70% notebooks, 30% datasets
  - Python dominates (85%)
  - Only 0.04% earn Gold medals

THE TIER EFFECT:
  - Grandmasters get ~5x more views than Novices
  - Higher tiers have significantly better medal rates
  - Followers strongly correlate with tier

MEDAL SUCCESS FACTORS:
  - Competition-related content is ~1.5x more likely to medal
  - Higher quality scores strongly predict medals
  - More visualizations correlate with better outcomes

TEMPORAL PATTERNS:
  - Older content accumulates more engagement
  - Updated content has higher medal rates
  - Consistent publishing helps build audience

FOR NOTEBOOKS:
  - GPU usage correlates with higher engagement
  - Higher markdown ratios (more documentation) help
  - pandas, numpy, matplotlib are most common libraries

FOR DATASETS:
  - Higher usability scores drive more downloads
  - CSV is the most popular format
  - CC0 and CC BY are preferred licenses
''')

---
## Thank You!

If you found this analysis helpful, please **upvote** this notebook!

Feel free to fork this notebook and extend the analysis. Some ideas:
- Build a medal prediction model
- Analyze title patterns using NLP
- Create an engagement forecasting model
- Cluster content by success patterns